# Process reviews JSONL

Reads raw gzipped JSONL review files, filters by date range / sample size / fields, and writes a clean intermediate CSV for `prepare-dataset.ipynb`.

In [1]:
import json
import gzip
from pathlib import Path
from typing import Any
import pandas as pd

In [2]:
# --- Config ---
CATEGORIES = ["Beauty_and_Personal_Care", "Clothing_Shoes_and_Jewelry"]
SAMPLE_SIZE: int | None = 100_000  # Per category; None = use all
START_DATE: str | None = "2020-01-01"
END_DATE: str | None = "2022-12-31"
MIN_RATING: int | None = None
FIELDS_TO_KEEP: list[str] = ["user_id", "parent_asin", "rating", "timestamp"]
DATA_DIR: str = "../data"
OUTPUT_FILE: str = "reviews.csv"

## Common utils

In [3]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for _, line in enumerate(f):
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

def _date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def load_reviews(
    categories: list[str], sample_size: int | None = None,
    start_date: str | None = None, end_date: str | None = None,
    min_rating: int | None = None, fields: list[str] | None = None,
) -> list[dict[str, Any]]:
    start_ts = _date_to_ms(start_date)
    end_ts = _date_to_ms(end_date)
    print(f"Filtering reviews with criteria: start_date={start_date}, end_date={end_date}, min_rating={min_rating}")

    reviews: list[dict[str, Any]] = []
    for cat in categories:
        cat_reviews: list[dict[str, Any]] = []

        path = f"{DATA_DIR}/{cat}.jsonl.gz"
        print(f"Loading reviews: {path}")
        for obj in stream_jsonl(path, fields=fields):
            ts: Any = obj.get("timestamp")
            rating: Any = obj.get("rating")
            if start_ts is not None and (ts is not None and ts < start_ts):
                continue
            if end_ts is not None and (ts is not None and ts > end_ts):
                continue
            if min_rating is not None and (rating is not None and rating < min_rating):
                continue
            obj["category"] = cat
            cat_reviews.append(obj)

            if sample_size is not None and len(cat_reviews) >= sample_size:
                print(f"Reached sample size limit ({sample_size} reviews). Stopping.")
                break

        reviews.extend(cat_reviews)
    return reviews

## Load and filter reviews

In [4]:
reviews = load_reviews(
    CATEGORIES, sample_size=SAMPLE_SIZE,
    start_date=START_DATE, end_date=END_DATE,
    min_rating=MIN_RATING, fields=FIELDS_TO_KEEP,
)
print(f"Loaded {len(reviews)} reviews")

Filtering reviews with criteria: start_date=2020-01-01, end_date=2022-12-31, min_rating=None
Loading reviews: ../data/Beauty_and_Personal_Care.jsonl.gz
Reached sample size limit (100000 reviews). Stopping.
Loading reviews: ../data/Clothing_Shoes_and_Jewelry.jsonl.gz
Reached sample size limit (100000 reviews). Stopping.
Loaded 200000 reviews


In [5]:
df = pd.DataFrame(reviews)
display(df.sample(5))
df.info()

,user_id,parent_asin,rating,timestamp,category
186068,AGTCOOFMG6IVUPXTUCN6S2RR3TJQ,B0BWQMYCCH,5.0,1645062160191,Clothing_Shoes_and_Jewelry
155241,AH4GETIUB2DLCAWSNNTFSZYCDK6A,B09T32VQH7,3.0,1647237728709,Clothing_Shoes_and_Jewelry
198513,AE23DSK7YN5HZ7JBTZBSQUO5WADA,B08RJRPGBF,2.0,1657312229734,Clothing_Shoes_and_Jewelry
142487,AHC7V3NWQS3DZWYH64JK3EWOUF7A,B074B2TRT9,1.0,1632411717120,Clothing_Shoes_and_Jewelry
10581,AFT7RCXOXYGKRCS3367HUFN7MMYA,B08QPVX43D,5.0,1666389134427,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   user_id      200000 non-null  object 
 1   parent_asin  200000 non-null  object 
 2   rating       200000 non-null  float64
 3   timestamp    200000 non-null  int64  
 4   category     200000 non-null  object 
dtypes: float64(1), int64(1), object(3)
memory usage: 7.6+ MB


## Export to CSV

In [6]:
output_path = Path(DATA_DIR)
output_path.mkdir(parents=True, exist_ok=True)
file_path = output_path / OUTPUT_FILE
df.to_csv(file_path, index=False)
print(f"Wrote {file_path} ({df.shape[0]:,} rows, {df.shape[1]} columns)")

Wrote ../data/reviews.csv (200,000 rows, 5 columns)
